# Observability — Tracing Agents with LangSmith

This notebook combines **Human-in-the-Loop** with **LangSmith observability** — a platform for tracing, debugging, and evaluating LangChain/LangGraph applications.

## Key concepts

- **LangSmith** – A cloud platform by the makers of LangChain. It captures every step of your agent's execution as a "run" that you can inspect in a visual UI: which tools were called, what the model produced, how long each step took, etc.
- **Tracing** – When `LANGSMITH_TRACING=true` is set, every agent invocation is automatically logged to LangSmith. No other code changes are needed.
- **`run_id`** – A unique identifier (UUID) you generate and pass per `.invoke()` call. You use this ID to attach feedback (ratings, comments) to a specific run in LangSmith.
- **`LangSmithClient.create_feedback()`** – Programmatically submit user feedback (e.g., a thumbs-up score of 1) for a specific run. This is useful for building evaluation datasets.
- **`HumanInTheLoopMiddleware`** – Used here for booking tools (flight, hotel) — same pattern as in `Human-in-the-Loop.ipynb`.

## What's new in this notebook

| Feature | Source |
|---|---|
| Human-in-the-Loop | Human-in-the-Loop.ipynb |
| LangSmith tracing | **New in this notebook** |
| `run_id` per invocation | **New in this notebook** |
| Programmatic feedback submission | **New in this notebook** |

## Prerequisites

You'll need:
- An `OPENAI_API_KEY` stored in Colab secrets
- A `LANGSMITH_API_KEY` stored in Colab secrets
- A `LANGSMITH_WORKSPACE_ID` stored in Colab secrets

In [ ]:
# Install required packages.
!pip install -q langchain langchain-openai

In [ ]:
import json    # For serializing tool output to a readable string
import os      # For setting environment variables (LangSmith config)
import uuid    # For generating unique run IDs per agent invocation

from IPython.display import HTML
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Interrupt
from langsmith import Client as LangSmithClient  # LangSmith SDK for submitting feedback
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# --- LangSmith tracing configuration ---
# Setting these environment variables enables automatic tracing.
# Every call to agent.invoke() will be logged to your LangSmith project.
os.environ["LANGSMITH_TRACING"] = "true"                                    # Turn on tracing
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')         # Your LangSmith API key
os.environ["LANGSMITH_WORKSPACE_ID"] = userdata.get('LANGSMITH_WORKSPACE_ID') # Your workspace
os.environ["LANGSMITH_PROJECT"] = "LangChain Memory & Human-in-the-Loop"   # Project name in LangSmith UI

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [ ]:
# --- Define three travel tools ---
# These tools simulate a real travel booking system.
# book_flight and book_hotel will require human approval before they execute.

@tool
def search_travel_options(destination: str) -> str:
    """Call this tool to search for flight and accommodation options for a trip."""
    # Hardcoded mock data — simulates a real travel API response.
    options = {
        "destination": destination,
        "flights": [
            {"flight_code": "LH170", "departure": "08:10", "price_eur": 189},
            {"flight_code": "FR402", "departure": "09:05", "price_eur": 129},
        ],
        "hotels": [
            {"hotel_name": "Alexander Hub Hotel", "price_eur": 176},
            {"hotel_name": "Spring View Stay", "price_eur": 144},
        ]
    }
    # Return the options as a formatted JSON string so the model can read them easily.
    return json.dumps(options, indent=2)

@tool
def book_flight(traveler_name: str, flight_code: str) -> str:
    """Call this tool to book a flight."""
    # In a real app this would call a booking API.
    return f"Booked flight {flight_code} for {traveler_name}"

@tool
def book_hotel(traveler_name: str, hotel_name: str, nights: int) -> str:
    """Call this tool to book a hotel."""
    return f"Booked {nights} night(s) at {hotel_name} for {traveler_name}"

In [ ]:
# Build the travel agent.
# Both book_flight and book_hotel require human approval before execution.
# search_travel_options does NOT require approval (it's read-only).
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    tools=[search_travel_options, book_flight, book_hotel],
    system_prompt="You are a travel operations assistant.",
    checkpointer=InMemorySaver(),  # Required for HITL — saves state between pause and resume
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                book_flight.name: True,  # Pause before every flight booking
                book_hotel.name: True    # Pause before every hotel booking
            }
        )
    ],
)

In [ ]:
# Thread configuration — identifies this specific trip planning session.
configurable = {
    "thread_id": "berlin_trip_1"
}

In [ ]:
# Generate a unique run_id for the FIRST invocation.
# This UUID is passed to LangSmith so we can attach feedback (ratings) to this specific run later.
first_run_id = uuid.uuid4()

# Invoke the agent. It will:
#   1. Call search_travel_options("Berlin") — no interrupt needed.
#   2. Choose the cheapest flight (FR402, €129) and cheapest hotel (Spring View Stay, €144).
#   3. Attempt to call book_flight — PAUSED by HumanInTheLoopMiddleware.
#   4. Return with __interrupt__ containing the pending booking requests.
berlin_trip = agent.invoke(
    input={
        "messages": [HumanMessage("Plan a one-night business trip to Berlin for Dana. Search options, book a morning flight, and reserve one hotel night for up to 150 EUR. If there are many possible options, take the initiative and select the most affordable one.")]
    },
    config={
        "configurable": configurable,
        "run_id": first_run_id  # Link this invocation to our run_id for LangSmith feedback
    }
)

In [ ]:
# Print the conversation so far and the pending interrupt requests.
# You should see two pending approvals: one for book_flight, one for book_hotel.
print_conversation(berlin_trip["messages"])
print_interrupts(berlin_trip.get("__interrupt__", []))

In [ ]:
# Generate a SEPARATE run_id for the second invocation (the resume call).
# LangSmith treats each .invoke() as its own "run". We track both so we can rate them independently.
second_run_id = uuid.uuid4()

# NOTE: We expect two blocked tool calls — one to book a flight, one to book a hotel.
# We approve BOTH at once by passing two decisions in the list.
# The order of decisions matches the order of interrupts.
berlin_trip = agent.invoke(
    input=Command(
        resume={
            "decisions": [
                { "type": "approve" },  # Approve the flight booking
                { "type": "approve" }   # Approve the hotel booking
            ]
        }
    ),
    config={
        "configurable": configurable,   # Same thread — resuming from where we paused
        "run_id": second_run_id          # New run_id for this continuation invocation
    }
)

In [ ]:
# Print the final conversation after both bookings were approved and executed.
print_conversation(berlin_trip["messages"])

In [ ]:
# Submit positive feedback (score=1 = thumbs up) for BOTH runs to LangSmith.
# This is how you programmatically rate agent runs, which is useful for:
#   - Building evaluation datasets
#   - Tracking quality over time as you iterate on your prompts/agent design
#   - Comparing different agent configurations
lang_smith_client = LangSmithClient()

# Attach a "user_rating" score of 1 (positive) to the first run (initial planning + interrupts)
_ = lang_smith_client.create_feedback(run_id=first_run_id, key="user_rating", score=1)

# Attach the same rating to the second run (the resume/approval run)
_ = lang_smith_client.create_feedback(run_id=second_run_id, key="user_rating", score=1)